# 面试问题：LLM KV Cache 量化怎样实现，怎样在显存、注意力误差与异常值之间取舍？

**一句话回答。** KV 量化不能只把张量转成低 bit：必须定义分组与 K/V 轴、scale/zero-point、异常值回退、模型与 RoPE 版本，并用全精度 attention 作为 oracle 同时报显存和误差。

本 Notebook 仅用 Python 标准库手写数据合同、核心算法和失败分支；受控样例用于验证不变量，不代表生产吞吐、模型质量或硬件精度。

**资料入口。** [KIVI](https://arxiv.org/abs/2402.02750) 讨论 K/V 的非对称量化；这里为了可审查，使用小向量而非 GPU bit packing。


In [ ]:
question = "KV Cache 量化"  # 执行本行的状态、计算或校验逻辑。
bits = 4  # 执行本行的状态、计算或校验逻辑。
assert bits in (2, 4, 8)  # 执行本行的状态、计算或校验逻辑。
assert "量化" in question  # 执行本行的状态、计算或校验逻辑。
assert 2 ** bits == 16  # 执行本行的状态、计算或校验逻辑。

## 1. 先量化已写入的 cache，而不是量化模型权重

KV Cache 是每层、每个 KV head、每个历史 token 的运行时状态。量化格式必须与模型 revision、RoPE 配置和 head shape 绑定；权重量化正确并不能保证 cache 量化正确。


In [ ]:
full_values = [-1.2, -0.2, 0.0, 0.7, 1.1]  # 执行本行的状态、计算或校验逻辑。
assert len(full_values) == 5  # 执行本行的状态、计算或校验逻辑。
assert min(full_values) < 0 < max(full_values)  # 执行本行的状态、计算或校验逻辑。
assert bits < 16  # 执行本行的状态、计算或校验逻辑。

## 2. 非对称 group-wise affine quantization

非对称量化把每个 group 的最小值映射到 code 0、最大值映射到最大 code。真实实现需要按论文选择 K/V 的轴和粒度；本例先把 scale 与 zero-point 的可复放合同写清。


In [ ]:
def quantize(values, bit_width):  # 执行本行的状态、计算或校验逻辑。
    levels = (1 << bit_width) - 1  # 执行本行的状态、计算或校验逻辑。
    low, high = min(values), max(values)  # 执行本行的状态、计算或校验逻辑。
    scale = (high - low) / levels if high != low else 1.0  # 执行本行的状态、计算或校验逻辑。
    codes = tuple(max(0, min(levels, round((value - low) / scale))) for value in values)  # 执行本行的状态、计算或校验逻辑。
    return {"codes": codes, "low": low, "scale": scale, "bits": bit_width}  # 执行本行的状态、计算或校验逻辑。
packed = quantize(full_values, bits)  # 执行本行的状态、计算或校验逻辑。
assert packed["codes"][0] == 0  # 执行本行的状态、计算或校验逻辑。
assert packed["codes"][-1] == 15  # 执行本行的状态、计算或校验逻辑。
assert packed["bits"] == 4  # 执行本行的状态、计算或校验逻辑。

## 3. 反量化与逐组元数据

运行 attention 时先反量化当前需要的 K/V block。生产可使用 fused kernel，教学实现保留每组 low/scale，避免把不同 token、head 或版本的 scale 混用。


In [ ]:
def dequantize(record):  # 执行本行的状态、计算或校验逻辑。
    return tuple(record["low"] + code_value * record["scale"] for code_value in record["codes"])  # 执行本行的状态、计算或校验逻辑。
recovered = dequantize(packed)  # 执行本行的状态、计算或校验逻辑。
max_error = max(abs(left - right) for left, right in zip(full_values, recovered))  # 执行本行的状态、计算或校验逻辑。
assert len(recovered) == len(full_values)  # 执行本行的状态、计算或校验逻辑。
assert max_error <= packed["scale"] / 2 + 1e-12  # 执行本行的状态、计算或校验逻辑。
assert recovered[0] == min(full_values)  # 执行本行的状态、计算或校验逻辑。

## 4. 把近似限制在可测的 attention 路径

只看单点误差不够：K 的量化改变 score，V 的量化改变加权输出。这里以稳定 softmax 和完整 cache attention 为 oracle，检查小量化误差没有改变受控样例的 top-1 key。


In [ ]:
import math  # 执行本行的状态、计算或校验逻辑。
def softmax(values):  # 执行本行的状态、计算或校验逻辑。
    offset = max(values)  # 执行本行的状态、计算或校验逻辑。
    weights = [math.exp(value - offset) for value in values]  # 执行本行的状态、计算或校验逻辑。
    total = sum(weights)  # 执行本行的状态、计算或校验逻辑。
    return [value / total for value in weights]  # 执行本行的状态、计算或校验逻辑。
query = [0.8, -0.4, 0.2, 0.1, 0.3]  # 执行本行的状态、计算或校验逻辑。
full_probs = softmax([left * right for left, right in zip(query, full_values)])  # 执行本行的状态、计算或校验逻辑。
quant_probs = softmax([left * right for left, right in zip(query, recovered)])  # 执行本行的状态、计算或校验逻辑。
assert full_probs.index(max(full_probs)) == quant_probs.index(max(quant_probs))  # 执行本行的状态、计算或校验逻辑。
assert abs(sum(quant_probs) - 1.0) < 1e-12  # 执行本行的状态、计算或校验逻辑。
assert len(full_probs) == len(quant_probs)  # 执行本行的状态、计算或校验逻辑。

## 5. 异常值单独保存，避免一个值拉大整个 group

异常值会放大 affine scale，使普通值的分辨率下降。工程上可按阈值保存少量 full-precision residual；它增加元数据和分支，但必须用端到端误差/延迟证明收益。


In [ ]:
def quantize_with_outlier(values, bit_width, threshold):  # 执行本行的状态、计算或校验逻辑。
    normal = [value for value in values if abs(value) <= threshold]  # 执行本行的状态、计算或校验逻辑。
    record = quantize(normal or [0.0], bit_width)  # 执行本行的状态、计算或校验逻辑。
    outliers = {index: value for index, value in enumerate(values) if abs(value) > threshold}  # 执行本行的状态、计算或校验逻辑。
    return record, outliers  # 执行本行的状态、计算或校验逻辑。
normal_record, outliers = quantize_with_outlier(full_values, bits, 1.0)  # 执行本行的状态、计算或校验逻辑。
assert outliers == {0: -1.2, 4: 1.1}  # 执行本行的状态、计算或校验逻辑。
assert normal_record["bits"] == 4  # 执行本行的状态、计算或校验逻辑。
assert len(outliers) == 2  # 执行本行的状态、计算或校验逻辑。

## 6. cache entry 必须绑定格式和版本

不能把某个 checkpoint、RoPE 或 group size 的 code 解释为另一个格式。entry 还要记录 logical position；旧 block 被 allocator 复用时，引用计数、版本和量化元数据必须一起更新。


In [ ]:
entry = {"model_revision": "m1", "rope_revision": "r1", "group_size": 5, "record": packed, "position": 17}  # 执行本行的状态、计算或校验逻辑。
def compatible(entry_value, model_revision, rope_revision, group_size):  # 执行本行的状态、计算或校验逻辑。
    return entry_value["model_revision"] == model_revision and entry_value["rope_revision"] == rope_revision and entry_value["group_size"] == group_size  # 执行本行的状态、计算或校验逻辑。
assert compatible(entry, "m1", "r1", 5)  # 执行本行的状态、计算或校验逻辑。
assert not compatible(entry, "m2", "r1", 5)  # 执行本行的状态、计算或校验逻辑。
assert not compatible(entry, "m1", "r2", 5)  # 执行本行的状态、计算或校验逻辑。

## 7. 估算节省时不要忘记 scale、zero 与异常值

理想 bit 数不是实际字节数：每组有 scale/zero，异常值还有 index/value。本例用简化字节账本演示为何短上下文或很小 group 可能得不偿失。


In [ ]:
def byte_cost(value_count, bit_width, group_count, outlier_count):  # 执行本行的状态、计算或校验逻辑。
    code_bytes = (value_count * bit_width + 7) // 8  # 执行本行的状态、计算或校验逻辑。
    metadata_bytes = group_count * 8  # 执行本行的状态、计算或校验逻辑。
    outlier_bytes = outlier_count * 8  # 执行本行的状态、计算或校验逻辑。
    return code_bytes + metadata_bytes + outlier_bytes  # 执行本行的状态、计算或校验逻辑。
fp16_bytes = len(full_values) * 2  # 执行本行的状态、计算或校验逻辑。
quant_bytes = byte_cost(len(full_values), bits, 1, len(outliers))  # 执行本行的状态、计算或校验逻辑。
assert fp16_bytes == 10  # 执行本行的状态、计算或校验逻辑。
assert quant_bytes == 27  # 执行本行的状态、计算或校验逻辑。
assert quant_bytes > fp16_bytes  # 执行本行的状态、计算或校验逻辑。

## 8. 评测、回退和发布门禁

应按上下文长度、层/head、任务和 hard case 报告 attention/输出差异、困惑度或任务分数、吞吐与峰值显存。超出误差预算、缺少 metadata 或命中异常格式时，明确回退 FP16，而不是静默读取错误 cache。


In [ ]:
def admission(entry_value, observed_error, budget):  # 执行本行的状态、计算或校验逻辑。
    return compatible(entry_value, "m1", "r1", 5) and observed_error <= budget  # 执行本行的状态、计算或校验逻辑。
assert admission(entry, max_error, 0.1)  # 执行本行的状态、计算或校验逻辑。
assert not admission(entry, max_error, 0.001)  # 执行本行的状态、计算或校验逻辑。
assert not admission({**entry, "rope_revision": "old"}, max_error, 0.1)  # 执行本行的状态、计算或校验逻辑。

## 面试收束

面试时先分清 KV 量化、权重量化和分页管理；再说明量化轴/粒度、scale/zero、异常值、解码 attention oracle、元数据版本与 FP16 回退。不要以某个小向量 top-1 未变宣称生产模型无损。
